<a href="https://colab.research.google.com/github/syedalijabir/math252-project/blob/main/fpdc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# 1 mount gDrive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# 2. Install R and Git tools in Colab

!apt-get update -qq
!apt-get install -y r-base git

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
r-base is already the newest version (4.5.3-1.2204.0).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.


In [13]:
# 3. Clone repo into Colab

%cd /content
!git clone https://github.com/syedalijabir/math252-project.git
%cd /content/math252-project

/content
Cloning into 'math252-project'...
remote: Enumerating objects: 706, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 706 (delta 8), reused 20 (delta 5), pack-reused 677 (from 2)
Receiving objects: 100% (706/706), 2.24 GiB | 69.18 MiB/s, done.
Resolving deltas: 100% (128/128), done.
Updating files: 100% (313/313), done.
/content/math252-project


In [14]:
# 4. Enable R in Colab

%load_ext rpy2.ipython

# If errors, run this:
# !pip install rpy2
# %load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [15]:
# 5. Install required R packages
%%R
packages <- c(
  "readr",
  "dplyr",
  "ggplot2",
  "cluster",
  "factoextra",
  "clusterCrit",
  "FPDclustering"
)

installed <- rownames(installed.packages())
to_install <- setdiff(packages, installed)

if (length(to_install) > 0) {
  install.packages(to_install, repos = "https://cloud.r-project.org")
}

lapply(packages, library, character.only = TRUE)

[[1]]
 [1] "FPDclustering" "mvtnorm"       "ThreeWay"      "clusterCrit"  
 [5] "factoextra"    "cluster"       "ggplot2"       "dplyr"        
 [9] "readr"         "tools"         "stats"         "graphics"     
[13] "grDevices"     "utils"         "datasets"      "methods"      
[17] "base"         

[[2]]
 [1] "FPDclustering" "mvtnorm"       "ThreeWay"      "clusterCrit"  
 [5] "factoextra"    "cluster"       "ggplot2"       "dplyr"        
 [9] "readr"         "tools"         "stats"         "graphics"     
[13] "grDevices"     "utils"         "datasets"      "methods"      
[17] "base"         

[[3]]
 [1] "FPDclustering" "mvtnorm"       "ThreeWay"      "clusterCrit"  
 [5] "factoextra"    "cluster"       "ggplot2"       "dplyr"        
 [9] "readr"         "tools"         "stats"         "graphics"     
[13] "grDevices"     "utils"         "datasets"      "methods"      
[17] "base"         

[[4]]
 [1] "FPDclustering" "mvtnorm"       "ThreeWay"      "clusterCrit"  
 [5] "factoex

In [16]:
# 6. Copy or link repo files into Drive

%%R
base_repo <- "/content/math252-project"
drive_out <- "/content/drive/MyDrive/math252-project-colab-output"

dir.create(drive_out, recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(drive_out, "output"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(drive_out, "output", "fpdc_cache"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(drive_out, "output", "fpdc_plots"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(drive_out, "output", "fpdc_results"), recursive = TRUE, showWarnings = FALSE)

cat("Repo:", base_repo, "\n")
cat("Drive output:", drive_out, "\n")

Repo: /content/math252-project 
Drive output: /content/drive/MyDrive/math252-project-colab-output 


In [17]:
# 7. Upload or copy embeddings into the Colab repo if needed

#check they exist:

%%R
embedding_paths <- c(
  "/content/math252-project/output/embeddings/user_embeddings_64.csv",
  "/content/math252-project/output/embeddings/user_embeddings_128.csv",
  "/content/math252-project/output/embeddings/user_embeddings_256.csv"
)

print(file.exists(embedding_paths))
print(embedding_paths)

[1] TRUE TRUE TRUE
[1] "/content/math252-project/output/embeddings/user_embeddings_64.csv" 
[2] "/content/math252-project/output/embeddings/user_embeddings_128.csv"
[3] "/content/math252-project/output/embeddings/user_embeddings_256.csv"


In [18]:
import psutil, os, multiprocessing

ram_gb = psutil.virtual_memory().total / 1e9
cpu_count = multiprocessing.cpu_count()

print(f"RAM: {ram_gb:.1f} GB")
print(f"CPU cores visible: {cpu_count}")

if ram_gb < 20:
    print("Not using a high-RAM runtime")
else:
    print("Using a high-RAM runtime")

RAM: 54.8 GB
CPU cores visible: 8
Using a high-RAM runtime


In [ ]:
%%R
library(readr)
library(FPDclustering)

base_repo <- "/content/math252-project"

embedding_paths <- c(
  file.path(base_repo, "output", "embeddings", "user_embeddings_64.csv"),
  file.path(base_repo, "output", "embeddings", "user_embeddings_128.csv"),
  file.path(base_repo, "output", "embeddings", "user_embeddings_256.csv")
)

for (embedding_path in embedding_paths) {
  if (!file.exists(embedding_path)) {
    cat("Missing file:", embedding_path, "\n")
    next
  }

  cat("\n==============================\n")
  cat("TuckerFactors for:", basename(embedding_path), "\n")
  cat("==============================\n")

  embeddings <- read_csv(embedding_path, show_col_types = FALSE)
  x <- scale(as.matrix(embeddings[, -1]))

  # Change 6 if want to inspect more candidate q values
  tf <- TuckerFactors(x, 7)

  print(tf)
}


TuckerFactors for: user_embeddings_64.csv 
1 % of the process completed


In [ ]:
%%R
run_mode <- "full"   # change to "full" later

if (run_mode == "test") {
  embedding_paths <- c("/content/math252-project/output/embeddings/user_embeddings_64.csv")
  k_range <- 2:4
  q_range <- 2:3
  maxiter_fpdc <- 10
} else if (run_mode == "full") {
  embedding_paths <- c(
    "/content/math252-project/output/embeddings/user_embeddings_64.csv",
    "/content/math252-project/output/embeddings/user_embeddings_128.csv",
    "/content/math252-project/output/embeddings/user_embeddings_256.csv"
  )
  k_range <- 2:10
  q_range <- 2:6
  maxiter_fpdc <- 20
} else {
  stop("run_mode must be 'test' or 'full'")
}

In [ ]:
%%R
library(readr)
library(dplyr)
library(cluster)
library(clusterCrit)
library(FPDclustering)

set.seed(42)

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
base_repo <- "/content/math252-project"
drive_out <- "/content/drive/MyDrive/math252-project-colab-output"

dir.create(file.path(drive_out, "output"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(drive_out, "output", "fpdc_cache"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(drive_out, "output", "fpdc_results"), recursive = TRUE, showWarnings = FALSE)

all_embedding_paths <- c(
  file.path(base_repo, "output", "embeddings", "user_embeddings_64.csv"),
  file.path(base_repo, "output", "embeddings", "user_embeddings_128.csv"),
  file.path(base_repo, "output", "embeddings", "user_embeddings_256.csv")
)

# --------------------------------------------------
# 2. Mode switch
# --------------------------------------------------
run_mode <- "test"   # change to "full" later

if (run_mode == "test") {
  embedding_paths <- all_embedding_paths[1]   # only 64 for testing
  k_range <- 2:4
  q_range <- 2:3
  maxiter_fpdc <- 10
} else if (run_mode == "full") {
  embedding_paths <- all_embedding_paths
  k_range <- 2:8
  q_range <- 2:5
  maxiter_fpdc <- 20
} else {
  stop("run_mode must be 'test' or 'full'")
}

cat("Run mode:", run_mode, "\n")
cat("Embeddings:\n")
print(embedding_paths)
cat("k range:", paste(range(k_range), collapse = " to "), "\n")
cat("q range:", paste(range(q_range), collapse = " to "), "\n")
cat("maxiter_fpdc:", maxiter_fpdc, "\n\n")

# --------------------------------------------------
# 3. Main FPDC loop
# --------------------------------------------------
all_results <- list()
overall_start <- Sys.time()

for (embedding_path in embedding_paths) {

  if (!file.exists(embedding_path)) {
    cat("Missing file:", embedding_path, "\n")
    next
  }

  embedding_start <- Sys.time()
  embedding_tag <- tools::file_path_sans_ext(basename(embedding_path))

  cat("\n========================================\n")
  cat("Processing:", embedding_tag, "\n")
  cat("========================================\n")

  full_cache_file <- file.path(
    drive_out, "output", "fpdc_cache",
    paste0("fpdc_full_result_", embedding_tag, ".rds")
  )

  if (file.exists(full_cache_file)) {
    cat("Loading full cached result for:", embedding_tag, "\n")
    all_results[[embedding_tag]] <- readRDS(full_cache_file)
    next
  }

  embeddings <- read_csv(embedding_path, show_col_types = FALSE)
  user_ids <- embeddings[[1]]
  embedding_matrix <- scale(as.matrix(embeddings[, -1]))

  dmat <- dist(embedding_matrix)

  results_grid <- expand.grid(k = k_range, q = q_range)
  results_grid$Silhouette <- NA_real_
  results_grid$CH <- NA_real_
  results_grid$ProbSilh <- NA_real_
  results_grid$runtime_sec <- NA_real_
  results_grid$status <- "ok"
  results_grid$notes <- NA_character_

  total_jobs <- nrow(results_grid)
  fpdc_models <- vector("list", total_jobs)

  for (i in seq_len(total_jobs)) {
    k_val <- results_grid$k[i]
    q_val <- results_grid$q[i]

    cat(sprintf("\n[%d/%d] %s | k=%d q=%d\n",
                i, total_jobs, embedding_tag, k_val, q_val))

    cache_file <- file.path(
      drive_out, "output", "fpdc_cache",
      paste0("fpdc_", embedding_tag, "_k", k_val, "_q", q_val, ".rds")
    )

    if (file.exists(cache_file)) {
      cat("Loading cached result\n")
      cached <- readRDS(cache_file)

      model <- cached$model
      avg_sil <- cached$Silhouette
      ch_val <- cached$CH
      prob_sil <- cached$ProbSilh

      results_grid$runtime_sec[i] <- cached$runtime_sec
      results_grid$status[i] <- cached$status
      results_grid$notes[i] <- cached$notes

    } else {
      cat("Running FPDC\n")
      t0 <- Sys.time()

      model <- tryCatch(
        FPDC(
          embedding_matrix,
          k_val,
          maxiter_fpdc,
          q_val,
          nu = 10
        ),
        error = function(e) {
          results_grid$status[i] <<- "error"
          results_grid$notes[i] <<- conditionMessage(e)
          NULL
        }
      )

      runtime_i <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
      results_grid$runtime_sec[i] <- runtime_i

      if (is.null(model)) {
        cat("FAILED:", results_grid$notes[i], "\n")
        next
      }

      labels <- model$label

      sil <- silhouette(labels, dmat)
      avg_sil <- mean(sil[, 3])

      ch_val <- intCriteria(
        traj = embedding_matrix,
        part = as.integer(labels),
        crit = "Calinski_Harabasz"
      )$calinski_harabasz

      prob_sil <- Silh(model$probability)

      saveRDS(
        list(
          model = model,
          Silhouette = avg_sil,
          CH = ch_val,
          ProbSilh = prob_sil,
          runtime_sec = runtime_i,
          status = results_grid$status[i],
          notes = results_grid$notes[i],
          k = k_val,
          q = q_val,
          embedding_path = embedding_path
        ),
        cache_file
      )

      cat(sprintf("Done in %.1f sec | Sil=%.5f | CH=%.5f | ProbSilh=%.5f\n",
                  runtime_i, avg_sil, ch_val, prob_sil))
    }

    results_grid$Silhouette[i] <- avg_sil
    results_grid$CH[i] <- ch_val
    results_grid$ProbSilh[i] <- prob_sil
    fpdc_models[[i]] <- model
  }

  valid_idx <- which(!is.na(results_grid$Silhouette))
  if (length(valid_idx) == 0) {
    cat("No valid models for:", embedding_tag, "\n")
    next
  }

  best_index <- valid_idx[which.max(results_grid$Silhouette[valid_idx])]
  best_params <- results_grid[best_index, , drop = FALSE]

  best_k <- best_params$k
  best_q <- best_params$q
  best_fpdc <- fpdc_models[[best_index]]
  best_labels <- best_fpdc$label

  cluster_df <- data.frame(
    UserID = user_ids,
    Cluster = best_labels
  )

  cluster_sizes <- table(cluster_df$Cluster)
  runtime_total_sec <- as.numeric(difftime(Sys.time(), embedding_start, units = "secs"))

  tmp_notes <- unique(na.omit(results_grid$notes))
  notes_text <- if (length(tmp_notes) > 0) paste(tmp_notes, collapse = " | ") else NA_character_

  summary_row <- data.frame(
    method = "FPDC",
    embedding = embedding_tag,
    best_k = best_k,
    best_q = best_q,
    silhouette = best_params$Silhouette,
    CH = best_params$CH,
    prob_silh = best_params$ProbSilh,
    cluster_size_min = min(cluster_sizes),
    cluster_size_max = max(cluster_sizes),
    cluster_size_sd = sd(as.numeric(cluster_sizes)),
    runtime_total_sec = runtime_total_sec,
    notes = notes_text
  )

  res <- list(
    embedding_path = embedding_path,
    embedding_tag = embedding_tag,
    results_grid = results_grid,
    best_index = best_index,
    best_params = best_params,
    best_k = best_k,
    best_q = best_q,
    best_fpdc = best_fpdc,
    best_labels = best_labels,
    cluster_df = cluster_df,
    runtime_total_sec = runtime_total_sec,
    summary_row = summary_row
  )

  all_results[[embedding_tag]] <- res

  write.csv(
    cluster_df,
    file = file.path(
      drive_out, "output", "fpdc_results",
      paste0("fpdc_clusters_", embedding_tag, "_k", best_k, "_q", best_q, ".csv")
    ),
    row.names = FALSE
  )

  write.csv(
    results_grid,
    file = file.path(
      drive_out, "output", "fpdc_results",
      paste0("fpdc_tuning_results_", embedding_tag, ".csv")
    ),
    row.names = FALSE
  )

  saveRDS(
    best_fpdc,
    file = file.path(
      drive_out, "output", "fpdc_cache",
      paste0("best_fpdc_", embedding_tag, "_k", best_k, "_q", best_q, ".rds")
    )
  )

  saveRDS(
    res,
    file = full_cache_file
  )

  cat("\nFinished:", embedding_tag, "\n")
  cat("Best k:", best_k, "| Best q:", best_q, "\n")
  cat("Best silhouette:", best_params$Silhouette, "\n")
  cat("Embedding runtime:", round(runtime_total_sec, 1), "sec\n")
}

# --------------------------------------------------
# 4. Best overall summary
# --------------------------------------------------
if (length(all_results) > 0) {

  all_best <- lapply(all_results, function(res) {
    data.frame(
      embedding = res$embedding_tag,
      best_k = res$best_k,
      best_q = res$best_q,
      Silhouette = res$best_params$Silhouette,
      CH = res$best_params$CH,
      ProbSilh = res$best_params$ProbSilh
    )
  })

  all_best_df <- bind_rows(all_best)
  all_summary_df <- bind_rows(lapply(all_results, function(x) x$summary_row))

  best_overall_index <- with(all_best_df, order(-Silhouette, -CH))[1]
  best_overall <- all_best_df[best_overall_index, , drop = FALSE]
  best_name <- best_overall$embedding
  best_res <- all_results[[best_name]]

  write.csv(
    all_best_df,
    file = file.path(
      drive_out, "output", "fpdc_results",
      "fpdc_best_overall_summary.csv"
    ),
    row.names = FALSE
  )

  write.csv(
    all_summary_df,
    file = file.path(
      drive_out, "output", "fpdc_results",
      "fpdc_summary_table.csv"
    ),
    row.names = FALSE
  )

  write.csv(
    best_res$cluster_df,
    file = file.path(
      drive_out, "output", "fpdc_results",
      paste0("fpdc_best_overall_clusters_", best_name,
             "_k", best_res$best_k, "_q", best_res$best_q, ".csv")
    ),
    row.names = FALSE
  )

  write.csv(
    best_res$results_grid,
    file = file.path(
      drive_out, "output", "fpdc_results",
      "fpdc_tuning_results_best_overall.csv"
    ),
    row.names = FALSE
  )

  cat("\n========================================\n")
  cat("BEST OVERALL RESULT\n")
  cat("========================================\n")
  print(best_overall)
}

cat("\nAll done.\n")
cat("Total runtime:",
    round(as.numeric(difftime(Sys.time(), overall_start, units = "mins")), 2),
    "minutes\n")
cat("Saved files in:\n", file.path(drive_out, "output", "fpdc_results"), "\n")